# Task 1 — Game Trees, Information Sets and Posterior Distributions (10%)

## Objective

By the end of this task you should be able to:

- Understand the structure of the extensive-form game tree for tiny_hana: decision nodes, chance nodes, terminal nodes, and information sets.
- Given an opponent **agent** and a Player 1 information set, compute the **posterior distribution over game states** consistent with that information set under that agent's strategy.

## Relevant Files

| File | Role |
|------|------|
| `engine/state.py` | Live mutable game state — `execute_action`, `undo_move`, `get_info_state`, `get_chance_actions_with_probs`, `get_legal_actions`, `to_key` |
| `engine/info_state.py` | `RoundInfoState` — the agent's information interface |
| `student_attempts/` | **Your implementation folder** — copy final code here |
| `tasks/task1.ipynb` | This notebook — run cells top to bottom |

> **Note:** This notebook contains inline stubs to experiment with. Once you are happy with your implementation, copy the final code into `student_attempts/task1.py`.

## Section 1 — The Extensive-Form Game Tree

An **extensive-form game** represents the full sequential structure of a game as a tree:

- **Decision nodes** — a player chooses an action. Labelled by the acting player.
- **Chance nodes** — nature randomly selects an outcome (e.g., dealing cards, drawing from the deck). Each outcome has a known probability.
- **Terminal nodes** — the game is over; each player receives a payoff.

Because tiny_hana has **imperfect information** (players cannot see each other's hands or reserved cards), decision nodes are grouped into **information sets**. Two nodes are in the same information set if the acting player *cannot distinguish between them* based on their observations.

The live engine (`engine/state.py`) lets you traverse the game tree using `execute_action` / `undo_move`. At any point, `state.current_player` tells you which kind of node you are at:

- `CHANCE_PLAYER` (-1) → chance node; enumerate outcomes with `get_chance_actions_with_probs()`.
- `0` or `1` → decision node for that player; enumerate moves with `get_legal_actions()`.
- `is_round_complete() == True` → terminal node.



You may traverse the game tree interactively by calling `interactive_game_tree_traversal()` in `utility.py`. Make sure you understand the differences between chance and player nodes, and know how to access the transition probabilities for the former.


## Section 2 — Information Sets

Two decision nodes $h$ and $h'$ are in the same information set $I$ for player $i$ if and only if player $i$ cannot distinguish $h$ from $h'$ using their observations.

For tiny_hana, the observable and hidden information at a decision node:

| Observable | Hidden |
|------------|--------|
| Your hand | Opponent's hand |
| Your collected / reserved cards | Opponent's reserved card suit |
| Opponent's collected cards | Undrawn deck card |
| Opponent's reserved **count** | — |
| Geisha favors, turn number | — |

The hidden information is exactly why multiple world states can share the same information set — a player sees the same hand and favor state but cannot know which specific card the opponent reserved or what remains in the deck.

### `RoundInfoState` as the information set identifier

In the live engine, a player's information set is represented by `RoundInfoState` — a frozen record of every observation event the player has received so far:

```python
info_state = state.get_info_state(player_id)

# Query observable fields:
hand, used      = info_state.get_current_hand_and_actions()
favors          = info_state.get_favors()
my_col, opp_col = info_state.get_collected_cards()
opp_res_count   = info_state.get_opponent_reserved_count()  # count only — suit hidden
```

Two `RoundInfoState` objects are **equal** (`==`) if and only if their event tuples match, meaning the player has observed identical histories. This equality test will come in handy in this Task.

## Section 3 — Tree Traversal: `count_leaves`

### The `execute_action` / `undo_move` pattern

A naive tree traversal would `deepcopy` the game state at every node. The live engine instead offers `undo_move()`, which reverses the most recent `execute_action()` call in O(1) time by replaying only the minimal per-action diff stored in an internal undo stack. This is significantly faster and avoids allocating a new state object at each recursion level.

```python
state.execute_action(action)
try:
    recurse(state)      # state is now the child node
finally:
    state.undo_move()   # always restored to parent — even on exception
```

The `compute_number_of_leaves()` in `utility.py` traverses the game tree and reports how many terminal nodes there are. Make sure you understand what is going on here.

## Section 4 — Agent Strategies and Expected Payoffs

### Agent strategies

An agent's strategy is a mapping from information set to action probabilities. In the engine this is expressed as:

```python
info_state = state.get_info_state(player_id)          # RoundInfoState
dist       = agent.get_action_distribution(info_state) # Dict[Action, float]
```

The returned dict maps each legal action to a probability (summing to 1). A deterministic agent returns `{chosen_action: 1.0}`; a mixed strategy assigns positive weight to several actions.

### Information states can be compared

Two `RoundInfoState` objects are **equal** (`==`) if and only if the player has seen exactly the same sequence of observation events — i.e., they belong to the same information set. No extra bookkeeping is needed:

```python
info_a = state.get_info_state(0)
info_b = state.get_info_state(0)  # same underlying state
assert info_a == info_b            # True — identical event histories
```

After executing and undoing a move, the info state is restored to its original value:

```python
state.execute_action(action)
state.undo_move()
assert state.get_info_state(0) == info_a  # True
```

This equality check is the key operation in `compute_posterior_given_opponent_agent` (Section 5).

### `compute_agent_payoffs` — weighted DFS

`count_leaves` (Section 3) branched uniformly over every action. 
| Node type | Branch weight |
|-----------|---------------|
| Chance | outcome probability from `get_chance_actions_with_probs()` |
| Player | action probability from `agent.get_action_distribution(info_state)` |
| Terminal | `terminal_payoff()` × accumulated probability |

Code to compute agent (1)'s payoffs is given in `utility.py`, given by `compute_agent_payoffs()`.

The result is Player 1's expected payoff under both agents' strategies. Read through the code and make sure you understand it.

## Section 5 — Posterior Distribution over States

### Motivation

Suppose you are Player 2 and you are at a particular information set $I$. You cannot observe the full game state — multiple states are consistent with your observations. Since Hanamijoki is perfect recall (you can take my word for it!), knowing that you are at $I$ essentially tells you exactly what you have played in the past. Therefore, if you also know something about *how the opponent plays* (their strategy $\sigma_0$), you can compute a probability distribution over the states consistent with $I$:

$$P(s \mid I, \sigma_0) \;\propto\; \pi^c(s) \cdot \pi^{\sigma_0}(s)$$

where $\pi^c(s)$ is the probability of the chance events leading to $s$ and $\pi^{\sigma_0}(s)$ is the probability of the opponent's actions leading to $s$. 

Apart from just being a useful academic exercise, knowing this posterior can be useful sometimes, potentially in finding the responses (in class, we did it via Treeplex traversals, which implicitly handles all the contribution from chance).

### Algorithm sketch (depth-first traversal from the initial state)

1. **Chance node:** branch over all chance outcomes; multiply the running probability by each outcome's probability.
2. **Opponent node (Player 1):** query the opponent agent's action distribution; branch and multiply by each action probability.
3. **Target-player node (Player 2):**
   - If `state.get_info_state(1) == target_info_state`: record `state.to_key() → prob` in the dictionary and **stop**.
   - Otherwise: recurse with probability weight unchanged (Player 1's own actions do not affect the posterior).

After traversal, normalise distributions in the dictionary so probabilities sum to 1.

We have included a function `generate_random_infoset()` that randomly samples an infoset in a game. You may use this to test your code.

## Task — Implement `compute_posterior_given_opponent_agent`

**Signature:**

```python
def compute_posterior_given_opponent_agent(initial_state: env.GameState, 
                      target_info_state: RoundInfoState,
                      opponent_agent: env.Agent) -> Dict[float]:
    
    """
    Given an initial game state and a target information state, as well as the opponent's strategy,
    compute the posterior distribution over the possible game states that are consistent with the target information state.

    Args:
        initial_state: GameState object representing the initial state of the game
        target_info_state: RoundInfoState object representing the information state for which we want to compute the posterior
        opponent_agent: Agent object representing the opponent's strategy
    
    Returns:
        A dictionary (or defaultdict) that maps `state key`s to the probability of that state being the true state given 
        the target information state and opponent strategy.

    Hint: 
        - For any state, use state.to_key() to obtain its `state key`.
        - Refer to `compute_number_of_leaves` and `compute_agent_payoffs` for examples of how to traverse the game tree. 
    """
```

**What it computes:** A probability distribution over `GameStateKey` values representing every concrete game state consistent with `target_info_state`, weighted by chance probabilities and the opponent's action probabilities.

_Hint_: You may use roughly the same skeleton as `count_leaves`.

Complete the function `compute_posterior_given_opponent_agent()` in `student_attempts/task1.py` and submit it on coursemology.

## Additional Notes/Hints
- Make sure to renormalize your probability vector! Posterior distributions are *still* distributions, so they must sum to 1.
- While we not technically grade you on efficiency for this task, your posterior distribuion computation should run in time that is *linear* in the size of the game tree. 